In [19]:
import pandas as pd
import numpy as np
from linearmodels import IV2SLS
import statsmodels.api as sm

import warnings
warnings.filterwarnings('ignore')

In [20]:
df = pd.read_excel('Individual Assignment Data File Electronic_sales.xlsx')
df.head()

,Customer ID,Age,Gender,Loyalty Member,Product Type,SKU,Rating,Order Status,Payment Method,Total Price,Unit Price,Quantity,Purchase Date,Shipping Type,Add-ons Purchased,Add-on Total
0,1000,53,Male,No,Smartphone,SKU1004,2,Cancelled,Credit Card,5538.33,791.19,7,2024-03-20,Standard,"Accessory,Accessory,Accessory",40.21
1,1000,53,Male,No,Tablet,SKU1002,3,Completed,Paypal,741.09,247.03,3,2024-04-20,Overnight,Impulse Item,26.09
2,1002,41,Male,No,Laptop,SKU1005,3,Completed,Credit Card,1855.84,463.96,4,2023-10-17,Express,NaN,0.00
3,1002,41,Male,Yes,Smartphone,SKU1004,2,Completed,Cash,3164.76,791.19,4,2024-08-09,Overnight,"Impulse Item,Impulse Item",60.16
4,1003,75,Male,Yes,Smartphone,SKU1001,5,Completed,Cash,41.50,20.75,2,2024-05-21,Express,Accessory,35.56


In [21]:
# Optional: You might want to filter only for "Completed" orders to measure actual sales
df = df[df['Order Status'] == 'Completed'].copy()


In [22]:
df['Gender'] = df['Gender'].map({'Male': 1.0, 'Female': 0.0}) 
df['Loyalty Member'] = df['Loyalty Member'].map({'No': 0.0, 'Yes': 1.0})

df['Purchase Date'] = pd.to_datetime(df['Purchase Date'], errors='coerce')
df['Month'] = df['Purchase Date'].dt.month.astype(str)
month_dummies = pd.get_dummies(df['Month'], prefix='Month', drop_first=True)
month_cols = month_dummies.columns.tolist()
df = pd.concat([df, month_dummies.astype(int)], axis=1)
df['DayOfWeek'] = df['Purchase Date'].dt.dayofweek # Monday=0, Sunday=6
df['Is_Weekend'] = df['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)

df.head()

,Customer ID,Age,Gender,Loyalty Member,Product Type,SKU,Rating,Order Status,Payment Method,Total Price,...,Month_2,Month_3,Month_4,Month_5,Month_6,Month_7,Month_8,Month_9,DayOfWeek,Is_Weekend
1,1000,53,1.0,0.0,Tablet,SKU1002,3,Completed,Paypal,741.09,...,0,0,1,0,0,0,0,0,5,1
2,1002,41,1.0,0.0,Laptop,SKU1005,3,Completed,Credit Card,1855.84,...,0,0,0,0,0,0,0,0,1,0
3,1002,41,1.0,1.0,Smartphone,SKU1004,2,Completed,Cash,3164.76,...,0,0,0,0,0,0,1,0,4,0
4,1003,75,1.0,1.0,Smartphone,SKU1001,5,Completed,Cash,41.50,...,0,0,0,1,0,0,0,0,1,0
5,1004,41,0.0,0.0,Smartphone,SKU1001,5,Completed,Credit Card,83.00,...,0,0,0,1,0,0,0,0,6,1


In [23]:
# create a new column for log of quantity sold and log of price
df['log_quantity'] = np.log(df['Quantity'])
df['log_price'] = np.log(df['Unit Price'])

# Add a constant for the regression
df['const'] = 1  

In [24]:
elasticities = {}

print("=== Cross-Sectional Price Elasticity Results ===")

for p_type in df['Product Type'].unique():
    df_subset = df[df['Product Type'] == p_type].copy()
    
    # Ensure sufficient data and price variation across the product category
    if len(df_subset) < 15 or df_subset['log_price'].std() == 0:
        continue
        
    # Define Target (y) and Features (X)
    y = df_subset['log_quantity']
    # We include log_price, baseline controls, quality (Rating), and seasonality (month_cols)
    X_cols = ['const', 'log_price', 'Loyalty Member', 'Rating', 'Age'] + month_cols
    X = df_subset[X_cols]
    
    try:
        # Fit standard OLS with Robust Standard Errors (HC3) to handle heteroskedasticity
        model = sm.OLS(y, X)
        results = model.fit(cov_type='HC3') 
        
        elasticities[p_type] = {
            'Elasticity (beta)': results.params['log_price'],
            'P-Value': results.pvalues['log_price'],
            'Observations': len(df_subset)
        }
        
    except Exception as e:
        print(f"Error for {p_type}: {e}")

# 7. Display Results cleanly
results_df = pd.DataFrame(elasticities).T
print(results_df)

=== Cross-Sectional Price Elasticity Results ===
            Elasticity (beta)   P-Value  Observations
Tablet              -0.018439  0.458380        2745.0
Laptop               0.275665  0.000394        2686.0
Smartphone           0.009700  0.310196        4004.0
Smartwatch          -0.043937  0.355116        2636.0
Headphones           0.119582  0.159818        1361.0
